# Visual Self-Sufficiency (VSS) 
## *Does this reel still make sense when the sound is off?*

This notebook builds an end-to-end pipeline for processing **100 raw Instagram Reels** and turning them into structured, interpretable signals about one core idea:

### **Sound-Off Completeness**
This is a term introduced in this project.

**Sound-Off Completeness** means:

> **How much of a reel’s message still survives when the audio is removed?**

It looks at the **full reel**, not just the first frame or a thumbnail moment. The idea is to measure whether a viewer can still understand the content’s **intent, message, emotion, and utility** with **zero audio**.

Why does this matter? Because a huge amount of reels are watched on mute. People scroll in public, at work, late at night, with device audio off, or simply out of habit. It also makes this problem naturally relevant for viewers who are **deaf or hard of hearing**. So this is not just a creative problem. It is also an **accessibility** problem and a **distribution** problem.

In this notebook, the final score we compute is called:

### **Visual Self-Sufficiency (VSS)**
Visual Self-Sufficiency is the actual score used to quantify Sound-Off Completeness.

Put simply:

> **VSS asks how much of the reel still works when the sound disappears.**

A high VSS score means the reel is doing a strong job visually. Even on mute, the viewer can still follow what is happening and why it matters. A low VSS score means the reel depends heavily on narration, spoken explanation, lyrics, or audio context.

---

## What this problem is really about

At its heart, this is a measure of **visual self-sufficiency**.

Think of reels as sitting on a spectrum.

At one end, you have something like a **dance reel built entirely around trending audio**. Remove the sound, and a lot of the meaning disappears. You still see movement, but the point of the content weakens fast.

At the other end, you have something like a **product unboxing or tutorial with strong on-screen text**. Mute it, and you still understand what is being shown, what the product is, and why the reel exists.

This project is about locating each reel on that spectrum.

---

## Why this is interesting beyond just scoring

A raw number is only useful if it leads to a real insight.

This becomes valuable in a few big ways:

### **1. It helps compare creators and content styles**
Two creators may have similar reach, but one may consistently produce content that still works beautifully on mute while the other depends almost entirely on sound. That difference matters.

### **2. It works like a creative quality check**
If a reel scores low, that can reveal what is missing:
- the product name is only spoken
- the key explanation is audio-only
- the visuals are engaging, but not informative enough on their own

That makes the score useful before launch, not just after the fact.

### **3. It surfaces a really interesting commercial tension**
Here is the inversion that makes this problem especially compelling:

- **Audio-dependent reels** are often easier to make trend
- **Visually self-sufficient reels** are often more useful to advertisers

That creates a tension between **virality** and **monetizability**.

A reel with 10 million views that only works with sound is a very different commercial asset from a reel with 500K views whose message is crystal clear on mute. This project shifts the conversation from:

> **How popular is this?**

to:

> **How usable is this when audio cannot be assumed?**

That is often the more valuable question.

---

## High-level approach

This notebook estimates VSS by looking at both sides of the reel:

- what the viewer **sees**
- what the viewer **hears**

The logic is:

- if visuals and audio are closely aligned, and the reel is not too speech-dependent, then the message is more likely to survive on mute
- if the audio is carrying unique meaning, especially spoken meaning, then muting the reel hurts much more

So the score is not just about whether audio exists. It is about whether the audio is **necessary**.

---

## Pipeline Overview

```text
Raw MP4
 ├──► Video Preprocessor ───► CLIP ViT-B/16 ───► E_visual (512-dim)
 │                                                     + ───────────────► Visual Self-Sufficiency Score
 └──► Audio Preprocessor ───► ESResNeXt ───────► E_audio  (512-dim)
                  │
                  └──► Audio-Type / BGM Detector ───► speech_ratio
                                                


## Setup Requirements

If you are running this notebook for the first time, follow these steps in order.

### 1. Create and activate a Python environment

#### Mac / Linux
```bash
python3 -m venv vssenv
source vssenv/bin/activate
python -m pip install --upgrade pip
pip install -r requirements.txt
python -m pip install "setuptools<82" wheel
python -m pip install --no-build-isolation visdom==0.2.4
```
#### Windows
```bash
python -m venv vssenv
vssenv\Scripts\activate
python -m pip install --upgrade pip
pip install -r requirements.txt
python -m pip install "setuptools<82" wheel
python -m pip install --no-build-isolation visdom==0.2.4
```


### 2. Verify that the Python dependencies installed correctly
``` bash
python -c "import torch, torchvision, numpy, pandas, cv2, PIL, matplotlib, seaborn, sklearn, ignite, termcolor, visdom, ftfy, tqdm, librosa, regex, ipykernel; print('All Python dependencies installed successfully.')"
```

### 3.  Check whether ffmpeg and ffprobe are installed

#### Mac / Linux
```bash 
which ffmpeg
which ffprobe
ffmpeg -version
ffprobe -version
```
#### Windows
```bash
where ffmpeg
where ffprobe
ffmpeg -version
ffprobe -version
```
### 4.  Install ffmpeg and ffprobe if they are missing

#### Mac (Homebrew)
```bash
brew install ffmpeg
```
#### Windows
Install FFmpeg and make sure it is added to your system PATH. Then re-run:
```bash
ffmpeg -version
ffprobe -version
```

### 5. Make sure the folder structure looks like this

- `project_root/`
  - `vss_pipeline.ipynb`
  - `requirements.txt`
  - `dataset/`
    - `reels/`
    - `metadata.csv`
  - `AudioCLIP/`
    - `model/`
    - `utils/`
    - `ignite_trainer/`
    - `assets/`
      - `AudioCLIP-Full-Training.pt`
      - `bpe_simple_vocab_16e6.txt.gz`

Note: The AudioCLIP file has been provided. 

### 6. Important note about the large AudioCLIP checkpoint file
The file AudioCLIP-Full-Training.pt is large, so it is not stored directly in the GitHub repository.

It has been uploaded separately as a Kaggle dataset:

Kaggle dataset link:
https://www.kaggle.com/datasets/soumyataneja/audioclip-full-training-pt

How to get it
1.	Open the Kaggle dataset link above.
2.	Download the dataset.

Where to place it
Move that file into the following folder inside the project:
```
AudioCLIP/assets/AudioCLIP-Full-Training.pt
```

So after placing it correctly, this path should exist:
```
project_root/AudioCLIP/assets/AudioCLIP-Full-Training.pt
```

This step is required for the notebook to load AudioCLIP successfully.



### 7. Final notes before running
•	All Python package versions are pinned in requirements.txt.  
•	tmp/ and output/ folders will be created automatically by the notebook.  
•	The notebook can run on CPU, but it will be slower.  
•	If the dependencies and folder structure are set up correctly, the notebook should run end-to-end without manual intervention.



## Setup & Imports

In [1]:
# Standard library
import json
import math
import os
import subprocess
import sys
import time
import warnings
import wave
from pathlib import Path
from typing import List, Optional, Tuple

# Data and media processing
import cv2
import numpy as np
import pandas as pd
from PIL import Image

# PyTorch
import torch
import torch.nn.functional as F
from torchvision import transforms

# Visualization
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import seaborn as sns

# Analysis
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Notebook display / plotting defaults
warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 130
plt.style.use("seaborn-v0_8-whitegrid")

# Device setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device   : {DEVICE}")
print(f"PyTorch  : {torch.__version__}")
print(f"OpenCV   : {cv2.__version__}")

Device   : cpu
PyTorch  : 2.6.0
OpenCV   : 4.10.0


## Configuration


In [2]:
from pathlib import Path
import os
import pandas as pd

# Base directory for the notebook run
PROJECT_DIR = Path(".").resolve()

# ── Input / output paths ──────────────────────────────────────────────────────
REELS_DIR = PROJECT_DIR / "dataset" / "reels"
METADATA_CSV = PROJECT_DIR / "dataset" / "metadata.csv"

# AudioCLIP is included in the provided data
AUDIOCLIP_DIR = PROJECT_DIR / "AudioCLIP"
AUDIOCLIP_WEIGHTS = AUDIOCLIP_DIR / "assets" / "AudioCLIP-Full-Training.pt"

TEMP_DIR = PROJECT_DIR / "tmp"
OUTPUT_DIR = PROJECT_DIR / "output"

TEMP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Video preprocessing config ────────────────────────────────────────────────
FRAME_SAMPLE_INTERVAL_S = 2.0
MIN_FRAMES = 4
MAX_FRAMES = 30
CLIP_INPUT_SIZE = 224

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]

# ── Audio preprocessing config ────────────────────────────────────────────────
AUDIO_SAMPLE_RATE = 44100
AUDIO_CHUNK_DURATION_S = 5.0
SAMPLES_PER_CHUNK = int(AUDIO_SAMPLE_RATE * AUDIO_CHUNK_DURATION_S)

BGM_SAMPLE_RATE = 16000
BGM_FRAME_SIZE = 512
BGM_HOP_SIZE = 256

# ── Model config ──────────────────────────────────────────────────────────────
VISUAL_BATCH_SIZE = 16
EMBEDDING_DIM = 512
SAVE_EMBEDDINGS = False

# ── VSS category thresholds ─────────────────────────────────────────────
VSS_THRESHOLDS = {
    "mute_safe": 0.82,
    "partially_mute_safe": 0.72,
}

# ── Quick verification ────────────────────────────────────────────────────────
reels = sorted(f for f in os.listdir(str(REELS_DIR)) if f.endswith(".mp4"))
metadata = pd.read_csv(str(METADATA_CSV))

print(f"Project root  : {PROJECT_DIR}")
print(f"Reels found   : {len(reels)}")
print(f"Metadata rows : {len(metadata)}")
print(f"AudioCLIP dir : {AUDIOCLIP_DIR.exists()}  ({AUDIOCLIP_DIR})")
print(f"Weights file  : {AUDIOCLIP_WEIGHTS.exists()}  ({AUDIOCLIP_WEIGHTS})")

metadata.head(3)

Project root  : /Users/st/Documents/Claude/Projects/Sparkonomy ML/vss_pipeline
Reels found   : 100
Metadata rows : 100
AudioCLIP dir : True  (/Users/st/Documents/Claude/Projects/Sparkonomy ML/vss_pipeline/AudioCLIP)
Weights file  : True  (/Users/st/Documents/Claude/Projects/Sparkonomy ML/vss_pipeline/AudioCLIP/assets/AudioCLIP-Full-Training.pt)


,filename,creator_id,shortcode
0,reel_00.mp4,62533072952,DL9UYHWP2SU
1,reel_01.mp4,65893188095,DVngdkFCKm5
2,reel_02.mp4,7875924727,DViqS4MirAC


##  Video Preprocessing 
Extracts frames from each MP4 at a fixed temporal interval and applies CLIP's normalization pipeline.

**Frame sampling:** 1 frame per 2 seconds (min 4, max 30) — captures visual content across the reel without redundant near-identical frames.  
**Transform:** Resize → CenterCrop(224) → Normalize(ImageNet) — matches CLIP ViT-B/16's expected input.


In [3]:
# CLIP image preprocessing for sampled video frames
clip_transform = transforms.Compose([
    transforms.Resize(CLIP_INPUT_SIZE,interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(CLIP_INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

def probe_video(video_path: str) -> Optional[dict]:
    """
    Extract basic video/audio metadata using ffprobe.

    Returns:
        dict with duration, file size, resolution, fps, and audio presence,
        or None if ffprobe fails.
    """
    cmd = [ "ffprobe","-v", "quiet","-print_format", "json","-show_format","-show_streams",str(video_path)]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        data = json.loads(result.stdout)
    except Exception:
        return None

    fmt = data.get("format", {})
    streams = data.get("streams", [])

    video_stream = next((s for s in streams if s.get("codec_type") == "video"), {})
    audio_stream = next((s for s in streams if s.get("codec_type") == "audio"), None)

    fps_str = video_stream.get("r_frame_rate", "30/1")
    try:
        num, den = fps_str.split("/")
        fps = float(num) / float(den)
    except Exception:
        fps = 30.0

    return {
        "duration": float(fmt.get("duration", 0)),
        "file_size_bytes": int(fmt.get("size", 0)),
        "width": int(video_stream.get("width", 0)),
        "height": int(video_stream.get("height", 0)),
        "fps": round(fps, 2),
        "has_audio": audio_stream is not None,
    }


def extract_frames(video_path: str) -> Tuple[List[Image.Image], List[int]]:
    """
    Sample frames from a video at fixed time intervals.

    Returns:
        raw_frames: list of PIL RGB images
        frame_indices: original frame indices sampled from the video
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_interval = max(1, int(video_fps * FRAME_SAMPLE_INTERVAL_S))
    target_indices = list(range(0, total_frames, frame_interval))

    # Ensure a minimum number of frames for very short videos
    if len(target_indices) < MIN_FRAMES and total_frames > 0:
        target_indices = np.linspace(0, total_frames - 1, MIN_FRAMES, dtype=int).tolist()

    # Cap the number of frames for efficiency
    if len(target_indices) > MAX_FRAMES:
        step = len(target_indices) / MAX_FRAMES
        target_indices = [target_indices[int(i * step)] for i in range(MAX_FRAMES)]

    target_set = set(int(idx) for idx in target_indices)
    raw_frames = []
    frame_indices = []

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx in target_set:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            raw_frames.append(Image.fromarray(rgb))
            frame_indices.append(frame_idx)

        frame_idx += 1

        if len(raw_frames) >= len(target_indices):
            break

    cap.release()
    return raw_frames, frame_indices


def preprocess_frames(raw_frames: List[Image.Image]) -> torch.Tensor:
    """
    Apply CLIP preprocessing to sampled frames.

    Returns:
        Tensor of shape (N, 3, CLIP_INPUT_SIZE, CLIP_INPUT_SIZE)
    """
    return torch.stack([clip_transform(img) for img in raw_frames])


def process_video(video_path: str) -> dict:
    """
    Full video preprocessing pipeline:
    1. Probe metadata
    2. Extract sampled frames
    3. Apply CLIP preprocessing
    """
    metadata = probe_video(video_path)
    if metadata is None:
        raise ValueError(f"ffprobe failed for: {video_path}")

    raw_frames, frame_indices = extract_frames(video_path)
    if not raw_frames:
        raise ValueError(f"No frames extracted from: {video_path}")

    frames_tensor = preprocess_frames(raw_frames)

    return {
        "frames_tensor": frames_tensor,
        "raw_frames": raw_frames,
        "frame_indices": frame_indices,
        "metadata": metadata,
        "n_frames": len(raw_frames),
    }

##  Audio Preprocessing
Extracts audio from each MP4 via ffmpeg and prepares it for AudioCLIP's ESResNeXt encoder.

**Two resolutions extracted:**
- 44.1 kHz (mono WAV) → chunked into 5-second windows for AudioCLIP embedding
- 16 kHz (mono WAV) → lightweight signal used by the BGM detector

**Language agnosticism:** Audio is processed acoustically as mel spectrograms, not transcribed — the pipeline works for any language.


In [4]:
def extract_audio_wav(video_path: str, output_wav: str, sr: int) -> bool:
    """
    Extract mono WAV audio from a video at the target sample rate.

    Returns:
        True if extraction succeeds, else False.
    """
    cmd = ["ffmpeg","-y","-i", str(video_path),"-ac", "1","-ar", str(sr),"-vn","-f", "wav",str(output_wav)]
    return subprocess.run(cmd, capture_output=True, timeout=60).returncode == 0


def load_wav(wav_path: str) -> Optional[np.ndarray]:
    """
    Load a WAV file as a float32 NumPy array normalized to [-1, 1].

    Returns:
        1D waveform array, or None if loading fails.
    """
    try:
        with wave.open(str(wav_path), "r") as wf:
            raw = wf.readframes(wf.getnframes())
            sample_width = wf.getsampwidth()

            if sample_width == 2:
                return np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
            elif sample_width == 4:
                return np.frombuffer(raw, dtype=np.int32).astype(np.float32) / 2147483648.0
            else:
                # Fallback for unexpected sample widths
                return np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    except Exception:
        return None


def chunk_audio(audio: np.ndarray) -> torch.Tensor:
    """
    Split waveform into fixed-length chunks for AudioCLIP.

    Returns:
        Tensor of shape (M, SAMPLES_PER_CHUNK)
    """
    if len(audio) == 0:
        return torch.zeros(1, SAMPLES_PER_CHUNK)

    if len(audio) < SAMPLES_PER_CHUNK:
        padded = np.zeros(SAMPLES_PER_CHUNK, dtype=np.float32)
        padded[:len(audio)] = audio
        return torch.from_numpy(padded).unsqueeze(0)

    n_chunks = len(audio) // SAMPLES_PER_CHUNK
    chunks = [
        audio[i * SAMPLES_PER_CHUNK:(i + 1) * SAMPLES_PER_CHUNK]
        for i in range(n_chunks)
    ]

    # Keeping the final partial chunk only if it is large enough to be useful
    remaining = len(audio) - n_chunks * SAMPLES_PER_CHUNK
    if remaining > SAMPLES_PER_CHUNK * 0.5:
        last = np.zeros(SAMPLES_PER_CHUNK, dtype=np.float32)
        last[:remaining] = audio[n_chunks * SAMPLES_PER_CHUNK:]
        chunks.append(last)

    return torch.from_numpy(np.stack(chunks))


def process_audio(video_path: str) -> dict:
    """
    Full audio preprocessing pipeline:
    1. Extract 44.1 kHz mono WAV for AudioCLIP
    2. Extract 16 kHz mono WAV for lightweight audio-type analysis
    3. Load waveform(s)
    4. Chunk 44.1 kHz audio for embedding
    """
    basename = os.path.splitext(os.path.basename(video_path))[0]
    wav_44k = str(TEMP_DIR / f"{basename}_44k.wav")
    wav_16k = str(TEMP_DIR / f"{basename}_16k.wav")

    empty = {
        "audio_chunks": torch.zeros(1, SAMPLES_PER_CHUNK),
        "raw_waveform_16k": np.array([], dtype=np.float32),
        "duration_s": 0.0,
        "n_chunks": 1,
        "has_audio": False,
    }

    # AudioCLIP input
    if not extract_audio_wav(video_path, wav_44k, AUDIO_SAMPLE_RATE):
        return empty

    audio_44k = load_wav(wav_44k)
    if audio_44k is None or len(audio_44k) == 0:
        return empty

    # Lightweight background-music / audio-type analysis input
    extract_audio_wav(video_path, wav_16k, BGM_SAMPLE_RATE)  # best-effort
    audio_16k = load_wav(wav_16k)
    if audio_16k is None:
        audio_16k = np.array([], dtype=np.float32)

    chunks = chunk_audio(audio_44k)

    return {
        "audio_chunks": chunks,
        "raw_waveform_16k": audio_16k,
        "duration_s": len(audio_44k) / AUDIO_SAMPLE_RATE,
        "n_chunks": chunks.shape[0],
        "has_audio": True,
    }



## AudioCLIP Embedder
Wraps the AudioCLIP model to produce `E_visual` and `E_audio` — both 512-dim L2-normalized unit vectors in a shared CLIP embedding space.

**Why AudioCLIP over alternatives:**
- CLIP's ViT-B/16 visual encoder reads **text overlays** in frames (unlike CAV-MAE)
- ESResNeXt processes **mel spectrograms acoustically** — language-agnostic, works for any spoken language
- Both modalities reside in the same 512-dim space, making cosine similarity between them meaningful


In [5]:
class AudioCLIPEmbedder:
    """
    Generate visual and audio embeddings in a shared AudioCLIP space.

    Visual encoder : CLIP ViT-B/16
    Audio encoder  : ESResNeXt
    Output         : 512-dim L2-normalized embeddings
    """

    def __init__(self, weights_path: str = None, device: str = None):
        self.device = device or DEVICE
        self.weights_path = Path(weights_path or AUDIOCLIP_WEIGHTS)
        self.model = None
        self._load_model()
    

    def _load_model(self) -> None:
        """
        Load AudioCLIP from either:
        1. a local source clone bundled with the notebook assets
        2. an installed package
        """
        # First try: local AudioCLIP source directory
        audioclip_dir = AUDIOCLIP_DIR
        if audioclip_dir.exists() and str(audioclip_dir) not in sys.path:
            sys.path.insert(0, str(audioclip_dir))

        try:
            from model import AudioCLIP as _AudioCLIP

            self.model = _AudioCLIP(pretrained=str(self.weights_path)).to(self.device).eval()
            print(f"[AudioCLIP] Loaded from local source  |  device={self.device}")
            print(f"            Weights: {self.weights_path.name}  ({self.weights_path.stat().st_size / 1e6:.0f} MB)")
            return
        except Exception as e:
            print(f"[AudioCLIP] Local source load failed: {e}")

        # Second try: installed package
        try:
            from audioclip import AudioCLIP as _AudioCLIP

            self.model = _AudioCLIP(pretrained=str(self.weights_path)).to(self.device).eval()
            print(f"[AudioCLIP] Loaded from installed package  |  device={self.device}")
            return
        except Exception as e:
            print(f"[AudioCLIP] Package load failed: {e}")

        raise RuntimeError(
            "AudioCLIP could not be loaded.\n"
            "Please ensure the following are available:\n"
            f"  AudioCLIP source : {AUDIOCLIP_DIR}\n"
            f"  Weights file     : {AUDIOCLIP_WEIGHTS}\n"
        )

    @torch.no_grad()
    def encode_visual(self, frames_tensor: torch.Tensor) -> torch.Tensor:
        """
        Encode sampled video frames into a single visual embedding.

        Returns:
            E_visual: (512,) L2-normalized tensor
        """
        frames_tensor = frames_tensor.to(self.device)
        all_embeddings = []

        for start in range(0, frames_tensor.shape[0], VISUAL_BATCH_SIZE):
            batch = frames_tensor[start:start + VISUAL_BATCH_SIZE]
            output = self.model.encode_image(batch)
            embedding = output[0].float() if isinstance(output, tuple) else output.float()
            all_embeddings.append(embedding)

        E_visual = torch.cat(all_embeddings, dim=0).mean(dim=0)
        return F.normalize(E_visual, p=2, dim=0).cpu()

    @torch.no_grad()
    def encode_audio(self, audio_chunks: torch.Tensor) -> torch.Tensor:
        """
        Encode chunked reel audio into a single audio embedding.

        Returns:
            E_audio: (512,) L2-normalized tensor
        """
        audio_chunks = audio_chunks.to(self.device)
        if audio_chunks.dim() == 1:
            audio_chunks = audio_chunks.unsqueeze(0)

        output = self.model.encode_audio(audio_chunks)
        embedding = output[0].float() if isinstance(output, tuple) else output.float()

        E_audio = embedding.mean(dim=0)
        return F.normalize(E_audio, p=2, dim=0).cpu()

    def encode_pair(
        self,
        frames_tensor: torch.Tensor,
        audio_chunks: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Convenience wrapper to encode both modalities.

        Returns:
            (E_visual, E_audio)
        """
        return self.encode_visual(frames_tensor), self.encode_audio(audio_chunks)



## Audio Type / Background Music Detector
Lightweight energy + zero-crossing rate (ZCR) heuristic operating on 16 kHz audio.

**Why this matters for VSS:** AudioCLIP embeds the full audio mixture. Background music creates an audio embedding that diverges from the visual embedding — artificially depressing VSS scores for visually self-sufficient reels. The BGM detector surfaces this via `speech_ratio`, which directly modulates the VSS formula.

**Speech vs music discrimination:**
- Speech → moderate ZCR (0.02–0.20), short bursts of energy
- Music → continuous energy, ZCR outside speech band
- Silence → low energy


## Why include a Background Music / Audio-Type Detector?

Because not all audio matters in the same way.

A reel with **spoken narration** usually loses much more meaning on mute than a reel with only **background music**. If we treated all audio the same, then a music-led reel and a speech-led reel could be penalized similarly, even though muting them affects understanding very differently.

That is why this pipeline includes a lightweight **audio-type detector**. Its role is to estimate whether the reel’s audio is mostly:
- **speech**
- **music**
- **mixed**
- **ambient**
- or **silence**

The most important signal it produces is **speech_ratio**.

This matters because the VSS formula should not over-penalize reels where the audio is mostly decorative background music. On the other hand, if the reel relies heavily on spoken explanation, muting it should reduce the score much more.

So in short, the audio-type detector helps answer a more nuanced version of the problem:

> **Is the audio carrying meaning, or just mood?**

That distinction makes the final VSS score much more realistic and much more useful.

In [6]:
def frame_audio(audio: np.ndarray, frame_size: int = 1024, hop_size: int = 512) -> np.ndarray:
    """
    Split a waveform into overlapping frames.

    Returns:
        Array of shape (num_frames, frame_size)
    """
    frames = []
    for start in range(0, max(1, len(audio) - frame_size + 1), hop_size):
        frames.append(audio[start:start + frame_size])

    if not frames:
        return np.zeros((0, frame_size), dtype=np.float32)

    return np.stack(frames)


def spectral_centroid(frame: np.ndarray, sr: int) -> float:
    """Compute spectral centroid for one frame."""
    magnitude = np.abs(np.fft.rfft(frame)) + 1e-8
    freqs = np.fft.rfftfreq(len(frame), d=1.0 / sr)
    return float((freqs * magnitude).sum() / magnitude.sum())


def spectral_flatness(frame: np.ndarray) -> float:
    """Compute spectral flatness for one frame."""
    magnitude = np.abs(np.fft.rfft(frame)) + 1e-8
    return float(np.exp(np.mean(np.log(magnitude))) / np.mean(magnitude))


def zero_crossing_rate(frame: np.ndarray) -> float:
    """Compute zero-crossing rate for one frame."""
    return float(np.mean(np.abs(np.diff(np.signbit(frame)))))


def rms_energy(frame: np.ndarray) -> float:
    """Compute RMS energy for one frame."""
    return float(np.sqrt(np.mean(frame ** 2) + 1e-10))


def detect_bgm(audio_16k: np.ndarray, sr: int = 16000) -> dict:
    """
    Classify reel audio into coarse audio types and detect background music.

    Output labels:
        - silence
        - speech
        - music
        - mixed
        - ambient

    Also returns speech/music/silence ratios and background-music presence.
    """
    empty = {
        "audio_type": "silence",
        "speech_ratio": 0.0,
        "music_ratio": 0.0,
        "silence_ratio": 1.0,
        "speech_present": False,
        "background_music_present": False,
    }

    if audio_16k is None or len(audio_16k) < 2048:
        return empty

    frames = frame_audio(audio_16k, frame_size=1024, hop_size=512)
    if len(frames) == 0:
        return empty

    # Frame-level features
    rms = np.array([rms_energy(frame) for frame in frames])
    zcr = np.array([zero_crossing_rate(frame) for frame in frames])
    centroid = np.array([spectral_centroid(frame, sr) for frame in frames])
    flatness = np.array([spectral_flatness(frame) for frame in frames])

    # Active vs silent frames
    energy_threshold = max(np.percentile(rms, 25) * 1.2, 0.005)
    active = rms > energy_threshold
    silence_ratio = float((~active).mean())

    if active.sum() == 0:
        return empty

    # Speech-like frames:
    # moderate ZCR, moderate centroid, lower flatness
    speech_like = (
        active
        & (zcr > 0.03) & (zcr < 0.12)
        & (centroid > 500) & (centroid < 2200)
        & (flatness < 0.25)
    )

    # Music-like frames:
    # more spectrally spread / brighter / noisier active audio
    music_like = (
        active
        & (
            (flatness >= 0.30)
            | (centroid >= 2000)
            | (zcr >= 0.15)
        )
    )

    # Ratios are computed over active frames only
    n_active = int(active.sum())
    speech_ratio = float(speech_like.sum() / n_active)
    music_ratio = float(music_like.sum() / n_active)

    # Presence flags
    speech_present = speech_ratio >= 0.05
    background_music_present = music_ratio >= 0.25

    # Final coarse label
    if silence_ratio >= 0.85:
        audio_type = "silence"
    elif speech_present and background_music_present:
        audio_type = "mixed"
    elif speech_present:
        audio_type = "speech"
    elif background_music_present:
        audio_type = "music"
    else:
        audio_type = "ambient"

    return {
        "audio_type": audio_type,
        "speech_ratio": round(speech_ratio, 4),
        "music_ratio": round(music_ratio, 4),
        "silence_ratio": round(silence_ratio, 4),
        "speech_present": speech_present,
        "background_music_present": background_music_present,
    }

##  VSS Computer — Speech-Weighted Composite Formula

```
alignment       = (1 + cosine_sim(E_visual, E_audio)) / 2    # ∈ [0, 1]
audio_weight    = 0.3 + 0.7 × speech_ratio                   # ∈ [0.3, 1.0]
audio_necessity = (1 - alignment) × audio_weight
VSS             = 1 - audio_necessity                         # ∈ [0, 1]
```
Examples:

| Scenario | alignment | speech_ratio | audio_weight | audio_necessity | VSS |
|---|---|---|---|---|---|
| Music video: BGM perfectly differs from visual | 0.5 | 0.0 | 0.30 | 0.15 | **0.85** |
| Tutorial: speech aligns with visual | 0.8 | 0.9 | 0.93 | 0.19 | **0.81** |
| Voiceover: speech diverges from visuals | 0.5 | 0.9 | 0.93 | 0.46 | **0.54** |
| Lecture: speech fully carries meaning, weak visual | 0.3 | 0.9 | 0.93 | 0.65 | **0.35** |




In [7]:
def compute_vss(
    E_visual: torch.Tensor,
    E_audio: torch.Tensor,
    speech_ratio: float = 0.5,
) -> dict:
    """
    Compute the Visual Self-Sufficiency (VSS) score.

    Args:
        E_visual: (512,) L2-normalized visual embedding
        E_audio: (512,) L2-normalized audio embedding
        speech_ratio: audio-type signal from the BGM detector
            0.0 = mostly music / non-speech
            1.0 = mostly speech

    Returns:
        Dictionary containing:
            - VSS score
            - VSS category
            - mute survival description
            - supporting diagnostics
    """
    # Cross-modal cosine similarity in [-1, 1]
    cross_modal_sim = float(torch.dot(E_visual, E_audio).clamp(-1.0, 1.0).item())

    # Map similarity to alignment in [0, 1]
    cross_modal_alignment = (1.0 + cross_modal_sim) / 2.0

    # Speech-heavy audio should matter more than decorative background music
    speech_ratio = max(0.0, min(1.0, float(speech_ratio)))
    speech_weight = 0.3 + 0.7 * speech_ratio

    # Estimate how necessary the audio is to preserve the reel's message
    audio_necessity = (1.0 - cross_modal_alignment) * speech_weight

    vss_score = 1.0 - audio_necessity
    vss_score = max(0.0, min(1.0, float(vss_score)))

    if vss_score >= VSS_THRESHOLDS["mute_safe"]:
        category = "mute_safe"
        mute_survival = "High — message survives well without audio"
    elif vss_score >= VSS_THRESHOLDS["partially_mute_safe"]:
        category = "partially_mute_safe"
        mute_survival = "Moderate — message mostly survives, but audio adds value"
    else:
        category = "audio_dependent"
        mute_survival = "Low — audio is important for understanding the message"

    return {
        "score": round(vss_score, 4),
        "category": category,
        "mute_survival": mute_survival,
        "cross_modal_sim": round(cross_modal_sim, 4),
        "cross_modal_alignment": round(cross_modal_alignment, 4),
        "speech_weight": round(speech_weight, 4),
    }



##  Pipeline Runner
Orchestrates the full end-to-end flow for each reel:


In [8]:
def process_single_reel(filename: str, embedder) -> Optional[dict]:
    """
    Process one reel end-to-end and return a flat result dictionary.

    Steps:
        1. Load metadata
        2. Preprocess video
        3. Preprocess audio
        4. Detect audio type / background music
        5. Compute AudioCLIP embeddings
        6. Compute VSS score and category
    """
    video_path = str(REELS_DIR / filename)
    if not os.path.exists(video_path):
        return None

    # ── Metadata lookup ───────────────────────────────────────────────────────
    row = metadata[metadata["filename"] == filename]
    meta_info = row.iloc[0].to_dict() if len(row) > 0 else {}

    record = {
        "filename": filename,
        "shortcode": meta_info.get("shortcode", ""),
        "creator_id": str(meta_info.get("creator_id", "")),
    }

    # ── Step 1: Video preprocessing ───────────────────────────────────────────
    try:
        video_data = process_video(video_path)
        video_meta = video_data["metadata"]

        record.update({
            "duration_s": video_meta.get("duration", 0),
            "width": video_meta.get("width", 0),
            "height": video_meta.get("height", 0),
            "fps": video_meta.get("fps", 0),
            "has_audio": video_meta.get("has_audio", False),
            "n_frames_sampled": video_data["n_frames"],
        })
    except Exception as e:
        print(f"  [ERR] Video: {e}")
        return None

    # ── Step 2: Audio preprocessing ───────────────────────────────────────────
    try:
        audio_data = process_audio(video_path)
    except Exception as e:
        print(f"  [WARN] Audio: {e}")
        audio_data = {
            "audio_chunks": torch.zeros(1, SAMPLES_PER_CHUNK),
            "raw_waveform_16k": np.array([], dtype=np.float32),
            "has_audio": False,
            "duration_s": 0.0,
            "n_chunks": 1,
        }

    record.update({
        "audio_duration_s": audio_data["duration_s"],
        "n_audio_chunks": audio_data["n_chunks"],
        "has_audio_content": audio_data["has_audio"],
    })

    # ── Step 3: Audio type / BGM detection ────────────────────────────────────
    bgm = detect_bgm(audio_data["raw_waveform_16k"])
    for key, value in bgm.items():
        record[f"bgm_{key}"] = value

    speech_ratio = bgm["speech_ratio"] if audio_data["has_audio"] else 0.0

    # ── Step 4: AudioCLIP embedding ───────────────────────────────────────────
    try:
        E_visual, E_audio = embedder.encode_pair(
            video_data["frames_tensor"],
            audio_data["audio_chunks"],
        )
    except Exception as e:
        print(f"  [ERR] Embedding: {e}")
        return None

    # ── Step 5: VSS computation ───────────────────────────────────────────────
    if not audio_data["has_audio"]:
        vss_result = {
            "score": 1.0,
            "category": "mute_safe",
            "mute_survival": "No audio track — fully understandable visually",
            "cross_modal_sim": 0.0,
            "cross_modal_alignment": 1.0,
            "speech_weight": 0.3,
        }
    else:
        vss_result = compute_vss(E_visual, E_audio, speech_ratio)

    for key, value in vss_result.items():
        record[f"vss_{key}"] = value

    # Optional: store raw embeddings
    if SAVE_EMBEDDINGS:
        record["e_visual"] = E_visual.numpy().tolist()
        record["e_audio"] = E_audio.numpy().tolist()

    return record


def run_pipeline(embedder, filenames: List[str] = None) -> List[dict]:
    """
    Run the full VSS pipeline across all reels.

    Returns:
        List of flat result dictionaries, one per successfully processed reel.
    """
    if filenames is None:
        filenames = sorted(f for f in os.listdir(str(REELS_DIR)) if f.endswith(".mp4"))

    n_files = len(filenames)
    start_time = time.time()

    print(f"\n{'═' * 72}")
    print(f"  VSS PIPELINE  —  Processing {n_files} reels")
    print(f"{'═' * 72}\n")
    print(f"{'#':>4}  {'File':<16}  {'VSS':>5}  {'Category':<22}  {'Audio':<8}  {'BGM?'}")
    print(f"{'─' * 72}")

    results = []

    for idx, filename in enumerate(filenames, start=1):
        reel_start = time.time()
        record = process_single_reel(filename, embedder)

        if record is not None:
            results.append(record)

            vss_score = record.get("vss_score", 0.0)
            category = record.get("vss_category", "?")
            audio_type = record.get("bgm_audio_type", "?")
            bgm_flag = "✓" if record.get("bgm_background_music_present", False) else " "
            elapsed = time.time() - reel_start

            print(
                f"{idx:>4}  {filename:<16}  {vss_score:>5.3f}  "
                f"{category:<22}  {audio_type:<8}  {bgm_flag}  ({elapsed:.1f}s)"
            )
        else:
            print(f"{idx:>4}  {filename:<16}  [FAILED]")

    total_time = time.time() - start_time
    avg_time = total_time / max(len(results), 1)

    print(f"{'═' * 72}")
    print(
        f"  Done: {len(results)}/{n_files} reels  |  "
        f"{total_time:.1f}s total  |  {avg_time:.1f}s per reel"
    )

    return results



##  Execute Pipeline
Initialize AudioCLIP and process all 100 reels end-to-end.


In [9]:
# Load AudioCLIP model
# Note: model weights are large (~537 MB), so initialization may take a short while.
embedder = AudioCLIPEmbedder()

# Process all reels and collect final outputs
results = run_pipeline(embedder)

print(f"\nTotal results collected: {len(results)}")

[AudioCLIP] Loaded from local source  |  device=cpu
            Weights: AudioCLIP-Full-Training.pt  (537 MB)

════════════════════════════════════════════════════════════════════════
  VSS PIPELINE  —  Processing 100 reels
════════════════════════════════════════════════════════════════════════

   #  File                VSS  Category                Audio     BGM?
────────────────────────────────────────────────────────────────────────
   1  reel_00.mp4       0.876  mute_safe               music     ✓  (1.5s)
   2  reel_01.mp4       0.862  mute_safe               music     ✓  (2.5s)
   3  reel_02.mp4       0.800  partially_mute_safe     mixed     ✓  (2.2s)
   4  reel_03.mp4       0.833  mute_safe               mixed     ✓  (1.7s)
   5  reel_04.mp4       0.864  mute_safe               music     ✓  (0.9s)
   6  reel_05.mp4       0.830  mute_safe               mixed     ✓  (2.2s)
   7  reel_06.mp4       0.767  partially_mute_safe     mixed     ✓  (1.1s)
   8  reel_07.mp4       0.834  mut

##  Generate Submission Files
Save three structured output files:
- `vss_scores.csv` — flat table, one row per reel, all signals
- `vss_detailed.json` — full detail per reel
- `vss_summary.json` — dataset-level statistics


In [10]:
# Flatten per-reel outputs into a tabular DataFrame
flat = [{k: v for k, v in r.items() if not isinstance(v, (list, dict))} for r in results]
df = pd.DataFrame(flat)

# Official submission columns
submission_cols = [
    "filename",
    "vss_score",
    "vss_category",
    "bgm_audio_type",
    "bgm_background_music_present",
]

submission_cols = [col for col in submission_cols if col in df.columns]

submission_df = df[submission_cols].copy().rename(columns={
    "bgm_audio_type": "audio_type",
    "bgm_background_music_present": "background_music_present",
})

# Save final CSV
csv_path = OUTPUT_DIR / "vss_scores.csv"
submission_df.to_csv(str(csv_path), index=False)
print(f"Submission CSV saved to: {csv_path}")
print(f"Rows: {len(submission_df)} | Columns: {len(submission_df.columns)}")

# Compact summary
cat_col = "vss_category"
n = len(submission_df)
cats = submission_df[cat_col].value_counts().to_dict()

print(f"\n{'═' * 60}")
print("  SUBMISSION SUMMARY")
print(f"{'═' * 60}")
for cat, cnt in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {cat:<24} {cnt:>3}  ({cnt/n*100:5.1f}%)")

print(f"\nBackground music present: {submission_df['background_music_present'].sum()} / {n}")

# Preview submission file
display(submission_df.head(10))

Submission CSV saved to: /Users/st/Documents/Claude/Projects/Sparkonomy ML/vss_pipeline/output/vss_scores.csv
Rows: 100 | Columns: 5

════════════════════════════════════════════════════════════
  SUBMISSION SUMMARY
════════════════════════════════════════════════════════════
  mute_safe                 52  ( 52.0%)
  partially_mute_safe       41  ( 41.0%)
  audio_dependent            7  (  7.0%)

Background music present: 91 / 100


,filename,vss_score,vss_category,audio_type,background_music_present
0,reel_00.mp4,0.8763,mute_safe,music,True
1,reel_01.mp4,0.8622,mute_safe,music,True
2,reel_02.mp4,0.8002,partially_mute_safe,mixed,True
3,reel_03.mp4,0.8327,mute_safe,mixed,True
4,reel_04.mp4,0.8643,mute_safe,music,True
5,reel_05.mp4,0.8297,mute_safe,mixed,True
6,reel_06.mp4,0.7674,partially_mute_safe,mixed,True
7,reel_07.mp4,0.8339,mute_safe,mixed,True
8,reel_08.mp4,0.8147,partially_mute_safe,mixed,True
9,reel_09.mp4,0.8369,mute_safe,mixed,True


## Closing Thought

The question this project set out to answer — **“How much of this reel’s message survives when the audio is removed?”** — sounds simple, but it gets at something genuinely difficult about multimodal content: visuals and audio do not carry meaning in the same way, and their relationship is rarely straightforward.

That is what makes this pipeline valuable. It does not reduce the problem to embedding similarity alone. By combining **cross-modal alignment** with **audio-type awareness**, it produces a signal that feels much closer to how people actually experience content on mute.

And while this notebook was built on 100 Instagram Reels, the idea is much bigger than this dataset. The same approach can extend to **ads, tutorials, branded content, creator videos, and short-form media across platforms and languages**. In that sense, VSS is not just a score — it is a way of turning unstructured video into a structured, interpretable signal that can support **accessibility, content strategy, creator feedback, ranking, and commercial decision-making**.

In short, the value is not just in the number. It is in what becomes possible once a video can be described in terms of **how it communicates, how it survives silence, and how usable it is in the real world**.